In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import KBinsDiscretizer
import warnings
warnings.filterwarnings('ignore')



dane = pd.read_csv("Titanic Dataset.csv")

dane['cabin_missing'] = dane['cabin'].isna().astype(int)

dane['title'] = dane['name'].str.extract(' ([A-Za-z]+)\.', expand=False)

dane['age'] = dane.groupby('title')['age'].transform(lambda x: x.fillna(x.median()))
dane=dane.drop(['body','boat','cabin','home.dest','ticket','name'],axis=1)



num_cols = dane.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    dane[col]=dane[col].fillna(dane[col].mean())

cat_cols = dane.select_dtypes(include=['object']).columns
for col in cat_cols:
    dane[col]=dane[col].fillna(dane[col].mode()[0])
    dane[col] = dane[col].astype('category').cat.codes


<>:15: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:15: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\kuba\AppData\Local\Temp\ipykernel_21964\716919972.py:15: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  dane['title'] = dane['name'].str.extract(' ([A-Za-z]+)\.', expand=False)


In [2]:


wyniki = {
    'Metoda': ['Równe części (cut)', 'Równoliczne części (qcut)', 'Klasteryzacja (KMeans)']
}

In [4]:
X = dane.drop('survived', axis=1)
y = dane['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



kolumny_numeryczne = ['age','fare']
k_vars = [2,5,10,25]

wyniki_dict = {
    "Metoda podziału": [
        "Równe części (uniform)", 
        "Równoliczne części (quantile)", 
        "Klasteryzacja (kmeans)"
    ]
}

for k in k_vars:
    result = []
    
    #===========Strategia 1 ============
    discretizer = KBinsDiscretizer(n_bins=k,encode='ordinal',strategy='uniform')
    X_train_k_uniform = X_train.copy()
    X_test_k_uniform = X_test.copy()
    X_train_k_uniform[kolumny_numeryczne] = discretizer.fit_transform(X_train_k_uniform[kolumny_numeryczne])
    X_test_k_uniform[kolumny_numeryczne] = discretizer.transform(X_test_k_uniform[kolumny_numeryczne])
    model_uniform = CategoricalNB()
    model_uniform.fit(X_train_k_uniform,y_train)
    result.append(model_uniform.score(X_test_k_uniform,y_test))
    
    #===========Strategia 2 ============
    discretizer = KBinsDiscretizer(n_bins=k,encode='ordinal',strategy='quantile')
    X_train_k_quantile = X_train.copy()
    X_test_k_quantile = X_test.copy()
    X_train_k_quantile[kolumny_numeryczne] = discretizer.fit_transform(X_train_k_quantile[kolumny_numeryczne])
    X_test_k_quantile[kolumny_numeryczne] = discretizer.transform(X_test_k_quantile[kolumny_numeryczne])
    model_quantile = CategoricalNB()
    model_quantile.fit(X_train_k_quantile,y_train)
    result.append(model_quantile.score(X_test_k_quantile,y_test))
    
    #===========Strategia 3 ============
    discretizer = KBinsDiscretizer(n_bins=k,encode='ordinal',strategy='kmeans')
    X_train_k_kmeans = X_train.copy()
    X_test_k_kmeans = X_test.copy()
    X_train_k_kmeans[kolumny_numeryczne] = discretizer.fit_transform(X_train_k_kmeans[kolumny_numeryczne])
    X_test_k_kmeans[kolumny_numeryczne] = discretizer.transform(X_test_k_kmeans[kolumny_numeryczne])
    model_quantile = CategoricalNB()
    model_quantile.fit(X_train_k_kmeans,y_train)
    result.append(model_quantile.score(X_test_k_kmeans,y_test))
    
    wyniki_dict[f"k={k}"] = [round(score, 4) for score in result]
    
df_wyniki = pd.DataFrame(wyniki_dict)
print(df_wyniki)    
    


                 Metoda podziału     k=2     k=5    k=10    k=25
0         Równe części (uniform)  0.7634  0.7710  0.7672  0.7672
1  Równoliczne części (quantile)  0.7901  0.7901  0.7557  0.7710
2         Klasteryzacja (kmeans)  0.7634  0.7672  0.7786  0.7748
